# Survey da aula: PKB com agentes

Análise inicial do questionário respondido pela turma em 2026-09-24 (25 respostas, `2026-09-24-survey.csv`).

**O que este notebook ensina**, além dos números:
1. escrever hipóteses *antes* de olhar os dados;
2. transformar respostas de formulário em dados analisáveis;
3. desconfiar de padrões em amostras pequenas (seção 6).

**Como rodar.** Precisa de Python com `pandas` e `matplotlib`. Com `uv`, da pasta deste notebook:

```
uv run --with jupyterlab --with pandas --with matplotlib jupyter lab
```

Ou com pip: `pip install pandas matplotlib jupyterlab` e `jupyter lab`.

**Dados anonimizados.** O export original tinha e-mail, nome, horários, gênero e faixa etária. Essas colunas foram removidas: com 25 pessoas, cruzar gênero ou idade com outras respostas identifica alguém (havia uma única pessoa em algumas categorias).

## 0. Hipóteses (escreva antes de rodar qualquer célula)

Edite esta célula. Não há resposta certa; o objetivo é registrar o que você esperava, para depois saber o que surpreendeu.

- Qual ferramenta a turma mais conhece? E a menos conhecida?
  - *minha aposta:*
- De 25 pessoas, quantas você acha que conseguem rodar um modelo de linguagem localmente (GPU ou Apple Silicon, RAM suficiente)?
  - *minha aposta:*
- A cor favorita de uma pessoa tem relação com a familiaridade dela com ferramentas de IA? Se sim, qual cor "sabe mais"?
  - *minha aposta:*

## 1. Carga e limpeza

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CSV = Path("2026-09-24-survey.csv")
FIG = Path("figuras")
FIG.mkdir(exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

raw = pd.read_csv(CSV, sep=";", encoding="utf-8-sig", dtype=str)
print(raw.shape)
raw.head(3)

O export do formulário tem nomes de coluna longos, escala de familiaridade em texto e a pergunta de múltipla escolha ("Antes da aula") numa única célula separada por `;`. Vamos:
- encurtar os nomes;
- converter a familiaridade para uma escala ordinal: `Nunca ouvi falar` = 0, `Ouvi falar, nunca usei` = 1, `Já usei` = 2;
- abrir o checklist em uma coluna verdadeiro/falso por item.

In [ ]:
PREFIXO = "Familiaridade com ferramentas."
ESCALA = {"Nunca ouvi falar": 0, "Ouvi falar, nunca usei": 1, "Já usei": 2}
NIVEIS = list(ESCALA)  # na ordem 0, 1, 2

FERRAMENTAS = {
    "GitHub / git": "git / GitHub",
    "Obsidian": "Obsidian",
    "Modelos locais (Ollama, vLLM, LM Studio)": "Modelos locais",
    "Gerenciadores de pacotes (pip, conda, npm, brew)": "Gerenc. de pacotes",
    "Jupyter Notebook": "Jupyter",
    "Ambientes científicos com IA (Claude Science, OpenScience)": "Ambientes científicos IA",
    "Interface de chat (ChatGPT, Claude)": "Chat de IA",
    "MCP e skills": "MCP e skills",
    "Workflows agênticos e automações": "Workflows agênticos",
    "opencode ou Claude Code (agentes no terminal)": "opencode / Claude Code",
}

df = pd.DataFrame({
    "so": raw["Sistema operacional principal"],
    "ram": raw["Memória RAM da máquina que você vai usar"],
    "gpu": raw["GPU da máquina que você vai usar"],
    "cor": raw["Entre estas, qual é a sua cor favorita?"],
})

fam_txt = raw[[PREFIXO + k for k in FERRAMENTAS]].copy()
fam_txt.columns = list(FERRAMENTAS.values())
fam = fam_txt.replace(ESCALA).astype(int)  # 0, 1, 2

checklist = (
    raw["Antes da aula (marque tudo que se aplica)"]
    .fillna("")
    .str.strip(";")
    .str.get_dummies(sep=";")
    .astype(bool)
)

print("Ferramentas:", list(fam.columns))
print("Itens do checklist:", list(checklist.columns))

In [ ]:
def tabela_md(t, indice=True):
    # Converte um DataFrame numa tabela Markdown, pronta para colar numa nota.
    t = t.reset_index() if indice else t
    linhas = ["| " + " | ".join(map(str, t.columns)) + " |",
              "|" + "---|" * len(t.columns)]
    linhas += ["| " + " | ".join(map(str, r)) + " |" for r in t.itertuples(index=False)]
    return "\n".join(linhas)

## 2. Familiaridade por ferramenta

Cada barra soma 100% das 25 respostas. Ordenadas pela proporção de "Já usei".

In [ ]:
contagem = (
    fam_txt.apply(lambda c: c.value_counts())
    .reindex(NIVEIS).fillna(0).astype(int).T
)
contagem = contagem.sort_values("Já usei")
prop = contagem.div(contagem.sum(axis=1), axis=0)

cores_niveis = ["#d9d9d9", "#9ecae1", "#2171b5"]
fig, ax = plt.subplots(figsize=(8, 5))
esquerda = np.zeros(len(prop))
for nivel, cor in zip(NIVEIS, cores_niveis):
    ax.barh(prop.index, prop[nivel], left=esquerda, color=cor, label=nivel)
    for i, (x, v) in enumerate(zip(esquerda, contagem[nivel])):
        if v >= 2:
            ax.text(x + prop[nivel].iloc[i] / 2, i, v, ha="center", va="center", fontsize=8)
    esquerda += prop[nivel].values
ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3, frameon=False)
ax.set_title("Familiaridade com ferramentas (n = 25)", pad=28)
fig.tight_layout()
fig.savefig(FIG / "familiaridade.png")
plt.show()

## 3. Um índice de familiaridade por pessoa

Somando a escala (0 a 2) nas 10 ferramentas, cada pessoa tem um índice de 0 a 20. A turma é homogênea ou há grupos?

In [ ]:
indice = fam.sum(axis=1)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(indice, bins=np.arange(-0.5, 21.5, 1), color="#2171b5", edgecolor="white")
ax.axvline(indice.median(), color="black", ls="--", lw=1, label=f"mediana = {indice.median():.0f}")
ax.set_xlabel("índice de familiaridade (0 a 20)")
ax.set_ylabel("pessoas")
ax.set_xticks(range(0, 21, 2))
ax.legend(frameon=False)
ax.set_title("Distribuição do índice de familiaridade")
fig.tight_layout()
fig.savefig(FIG / "indice.png")
plt.show()

indice.describe().round(1)

## 4. Quem consegue rodar um modelo local?

Classificação **por regra**, não por medição. É uma simplificação discutível (a VRAM da GPU importa mais do que a RAM, e não foi perguntada):
- **local viável**: GPU NVIDIA ou Apple Silicon, e 16 GB de RAM ou mais;
- **limítrofe**: 16 GB ou mais sem GPU dedicada (ou com AMD): roda modelos pequenos, devagar;
- **só terceiros**: 8 GB ou menos, ou não sabe a RAM.

In [ ]:
ORDEM_RAM = ["8 GB ou menos", "16 GB", "32 GB", "64 GB ou mais", "Não sei"]
ORDEM_GPU = ["NVIDIA", "Apple Silicon (M1 a M4)", "AMD", "Nenhuma ou não sei"]
CATEGORIAS = ["local viável", "limítrofe", "só terceiros"]
CORES_CAT = {"local viável": "#41ab5d", "limítrofe": "#fec44f", "só terceiros": "#ef6548"}


def viabilidade(ram, gpu):
    if ram in ("8 GB ou menos", "Não sei"):
        return "só terceiros"
    if gpu in ("NVIDIA", "Apple Silicon (M1 a M4)"):
        return "local viável"
    return "limítrofe"


df["inferencia"] = [viabilidade(r, g) for r, g in zip(df["ram"], df["gpu"])]

matriz = pd.crosstab(df["ram"], df["gpu"]).reindex(index=ORDEM_RAM, columns=ORDEM_GPU, fill_value=0)

fig, ax = plt.subplots(figsize=(8, 4))
for i, ram in enumerate(ORDEM_RAM):
    for j, gpu in enumerate(ORDEM_GPU):
        cor = CORES_CAT[viabilidade(ram, gpu)]
        n = matriz.loc[ram, gpu]
        ax.add_patch(plt.Rectangle((j, i), 1, 1, color=cor, alpha=0.85 if n else 0.2))
        if n:
            ax.text(j + 0.5, i + 0.5, n, ha="center", va="center", fontsize=12, weight="bold")
ax.set_xlim(0, len(ORDEM_GPU)); ax.set_ylim(len(ORDEM_RAM), 0)
ax.set_xticks(np.arange(len(ORDEM_GPU)) + 0.5, ["NVIDIA", "Apple Silicon", "AMD", "Nenhuma\nou não sei"])
ax.set_yticks(np.arange(len(ORDEM_RAM)) + 0.5, ORDEM_RAM)
ax.set_xlabel("GPU"); ax.set_ylabel("RAM")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in CORES_CAT.values()]
ax.legend(handles, CATEGORIAS, loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False)
ax.set_title("RAM × GPU: onde dá para rodar modelo local (n por célula)")
fig.tight_layout()
fig.savefig(FIG / "inferencia.png")
plt.show()

pd.crosstab(df["so"], df["inferencia"], margins=True, margins_name="total").reindex(columns=CATEGORIAS + ["total"])

## 5. As respostas são coerentes entre si?

Duas perguntas do formulário medem quase a mesma coisa por caminhos diferentes. Se alguém marcou no checklist "Já instalei o Ollama e baixei um modelo", esperamos "Já usei" em *Modelos locais*. Divergências não são erro do aluno: mostram como cada pessoa interpretou a pergunta, e isso é um viés de medida.

In [ ]:
pares = [
    ("Já instalei o Ollama e baixei um modelo", "Modelos locais"),
    ("Já usei o opencode", "opencode / Claude Code"),
]
for item, ferramenta in pares:
    t = pd.crosstab(checklist[item].map({True: "marcou", False: "não marcou"}),
                    fam_txt[ferramenta]).reindex(columns=NIVEIS, fill_value=0)
    t.index.name = f"checklist: {item}"
    t.columns.name = f"familiaridade: {ferramenta}"
    display(t)
    incoerentes = (checklist[item] & (fam[ferramenta] < 2)).sum()
    print(f"marcou o item mas não 'Já usei' em {ferramenta}: {incoerentes}\n")

Note que *opencode / Claude Code* inclui o Claude Code: quem usou só o Claude Code marca "Já usei" na familiaridade sem marcar "Já usei o opencode" no checklist. A direção inversa (marcou o opencode e não "Já usei") é que seria incoerente.

## 6. Cor favorita

In [ ]:
HEX = {"Azul": "#1f5fbf", "Preto": "#222222", "Verde": "#2e9e44", "Roxo": "#7b3fa0",
       "Vermelho": "#d62728", "Rosa": "#f06eaa", "Laranja": "#f28e1c",
       "Amarelo": "#f2c80f", "Branco": "#f4f4f4", "Cinza": "#8c8c8c", "Marrom": "#8c564b"}

n_cor = df["cor"].value_counts()

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(n_cor.index, n_cor.values, color=[HEX.get(c, "#999999") for c in n_cor.index], edgecolor="#555555")
for i, v in enumerate(n_cor.values):
    ax.text(i, v + 0.2, v, ha="center")
ax.set_ylabel("pessoas")
ax.set_title("Cor favorita (n = 25)")
fig.tight_layout()
fig.savefig(FIG / "cores.png")
plt.show()

### 6.1 Cor favorita × familiaridade: o padrão que aparece

Pergunta (absurda de propósito): **quem gosta de cada cor conhece mais ou menos as ferramentas?**

Para cada cor e cada ferramenta, calculamos a familiaridade média de quem escolheu aquela cor e subtraímos a média da turma naquela ferramenta. Azul = acima da média, vermelho = abaixo. O número entre parênteses é quantas pessoas escolheram a cor.

In [ ]:
media_turma = fam.mean()
media_cor = fam.groupby(df["cor"]).mean().loc[n_cor.index]
desvio = media_cor - media_turma

fig, ax = plt.subplots(figsize=(10, 4.5))
lim = np.abs(desvio.values).max()
im = ax.imshow(desvio.values, cmap="RdBu", vmin=-lim, vmax=lim, aspect="auto")
ax.set_xticks(range(len(desvio.columns)), desvio.columns, rotation=35, ha="right")
ax.set_yticks(range(len(desvio.index)), [f"{c} ({n_cor[c]})" for c in desvio.index])
for i in range(desvio.shape[0]):
    for j in range(desvio.shape[1]):
        v = desvio.iat[i, j]
        ax.text(j, i, "0" if abs(v) < 0.05 else f"{v:+.1f}", ha="center", va="center", fontsize=8,
                color="white" if abs(v) > lim * 0.6 else "black")
fig.colorbar(im, ax=ax, label="desvio da média da turma (escala 0 a 2)")
ax.set_title("Familiaridade por cor favorita, em relação à média da turma")
fig.tight_layout()
fig.savefig(FIG / "cor-x-familiaridade.png")
plt.show()

i, j = np.unravel_index(np.abs(desvio.values).argmax(), desvio.shape)
cor_m, ferr_m, v_m = desvio.index[i], desvio.columns[j], desvio.iat[i, j]
print(f"Manchete: quem prefere {cor_m} tem familiaridade com {ferr_m} "
      f"{abs(v_m):.1f} ponto(s) {'acima' if v_m > 0 else 'abaixo'} da média da turma!")
print(f"...com base em {n_cor[cor_m]} pessoa(s).")

Antes de seguir, olhe o gráfico e anote: qual padrão você contaria para alguém? Agora olhe o número entre parênteses da linha onde ele está.

### 6.2 Funil: desvios grandes vêm de grupos pequenos

Cada ponto é uma célula do gráfico anterior (uma cor × uma ferramenta). No eixo x, quantas pessoas escolheram aquela cor. Se a cor não tivesse nada a ver com a familiaridade (e não tem), o tamanho do desvio dependeria só do tamanho do grupo: grupos de 1 pessoa variam muito, grupos de 10 quase nada.

A faixa cinza mostra isso por simulação: **embaralhamos as cores entre as pessoas** 2.000 vezes, mantendo as respostas de familiaridade, e medimos até onde o desvio vai só por acaso (95% das vezes).

In [ ]:
rng = np.random.default_rng(42)
N_PERM = 2000
cores_obs = df["cor"].to_numpy()
F = fam.to_numpy(dtype=float)
media = F.mean(axis=0)
tamanhos = sorted(n_cor.unique())


def desvios_por_tamanho(cores):
    # Para cada cor: |média do grupo - média da turma| em cada ferramenta.
    out = {}
    for c in np.unique(cores):
        g = F[cores == c]
        out.setdefault(len(g), []).extend(np.abs(g.mean(axis=0) - media))
    return out


nulo = {n: [] for n in tamanhos}
for _ in range(N_PERM):
    for n, v in desvios_por_tamanho(rng.permutation(cores_obs)).items():
        nulo[n].extend(v)
p95 = pd.Series({n: np.percentile(nulo[n], 95) for n in tamanhos})

obs = desvios_por_tamanho(cores_obs)
fig, ax = plt.subplots(figsize=(7, 4))
ax.fill_between(p95.index, 0, p95.values, color="#e0e0e0", label="95% do que o acaso produz")
ax.plot(p95.index, p95.values, "o--", color="#888888", ms=4)
for n, v in obs.items():
    x = n + rng.uniform(-0.15, 0.15, len(v))
    ax.scatter(x, v, s=25, color="#2171b5", alpha=0.7)
ax.scatter([], [], color="#2171b5", label="células observadas")
ax.set_xticks(tamanhos)
ax.set_xlabel("pessoas que escolheram a cor")
ax.set_ylabel("|desvio da média da turma|")
ax.legend(frameon=False)
ax.set_title("Desvios grandes aparecem onde há poucas pessoas")
fig.tight_layout()
fig.savefig(FIG / "funil.png")
plt.show()

### 6.3 Teste de permutação: o padrão inteiro é maior que o acaso?

Uma medida única para o quanto a cor "explica" a familiaridade: o **η² médio** (eta ao quadrado), a fração da variação de cada ferramenta que fica entre os grupos de cor, média entre as ferramentas. Vale 0 se a cor não explica nada e 1 se explica tudo.

Comparamos o η² real com o η² de 2.000 turmas embaralhadas. O **p-valor** é a fração das turmas embaralhadas com η² maior ou igual ao real.

In [ ]:
def eta2_medio(cores):
    total = ((F - media) ** 2).sum(axis=0)
    entre = sum(
        (cores == c).sum() * (F[cores == c].mean(axis=0) - media) ** 2
        for c in np.unique(cores)
    )
    validas = total > 0  # ferramentas em que todos responderam igual não variam
    return (entre[validas] / total[validas]).mean()


eta_obs = eta2_medio(cores_obs)
eta_nulo = np.array([eta2_medio(rng.permutation(cores_obs)) for _ in range(N_PERM)])
p = (eta_nulo >= eta_obs).mean()

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(eta_nulo, bins=40, color="#cccccc", edgecolor="white", label="turmas embaralhadas")
ax.axvline(eta_obs, color="#d62728", lw=2, label=f"turma real (η² = {eta_obs:.2f})")
ax.set_xlabel("η² médio entre ferramentas")
ax.set_ylabel("simulações")
ax.legend(frameon=False)
ax.set_title(f"A cor explica a familiaridade? p = {p:.2f}")
fig.tight_layout()
fig.savefig(FIG / "permutacao.png")
plt.show()

print(f"η² real: {eta_obs:.3f} | η² médio só por acaso: {eta_nulo.mean():.3f} | p = {p:.2f}")

### 6.4 O que levar daqui

- **Não existe relação entre cor favorita e familiaridade com ferramentas**, e os dados são compatíveis com isso: o padrão real está dentro do que o embaralhamento produz.
- Mesmo assim, o primeiro gráfico parecia dizer algo. Com 25 pessoas e 7 cores, três cores têm **uma pessoa só**: a "média" delas é a resposta de um indivíduo, e sempre vai parecer extrema.
- O η² do acaso não é zero: com 7 grupos em 25 pessoas, dividir em grupos **sempre** "explica" uma parte da variação. Por isso se compara com o acaso, não com zero.
- **Comparações múltiplas**: o gráfico 6.1 tem 70 células. Se você procurar a mais extrema, sempre vai achar uma. A manchete da seção 6.1 foi escolhida assim.

Pergunta de bolso antes de acreditar num padrão: **quantas pessoas sustentam este número, e quantos números eu olhei até achar este?**

Volte à célula de hipóteses e compare com a sua aposta.

## 7. Tabelas para a nota

Saída em Markdown, para colar na nota da base.

In [ ]:
print(tabela_md(contagem.iloc[::-1].rename_axis("Ferramenta")))
print()
print(tabela_md(df["inferencia"].value_counts().reindex(CATEGORIAS).rename("pessoas").rename_axis("Inferência")))
print()
print(tabela_md(checklist.sum().sort_values(ascending=False).rename("pessoas").rename_axis("Antes da aula")))

## 8. Do notebook para a nota

O notebook é a **fonte reprodutível**; a nota é o **entendimento**, com as suas palavras. Não converta um no outro.

1. Na sua base, crie a pasta `03 Resources/Aulas/2026-09-24 - Survey PKB com agentes/` e copie para ela este notebook, o CSV e a pasta `figuras/`.
2. Peça ao assistente para ingerir o survey. A nota (`tipo: leitura`) deve ter no frontmatter `fonte: "[[2026-09-24-survey.csv]]"` e `analise: "[[2026-09-24-analise-survey.ipynb]]"`, embutir as figuras (`![[figuras/familiaridade.png]]`) e usar as tabelas da seção 7.
3. Inclua as suas hipóteses da seção 0 e o que surpreendeu.
4. Padrões entre variáveis entram como pergunta (`> [!gap]`), não como achado. A seção 6 explica por quê.
5. Seção "Fora do resumo": o que você não analisou (ex.: as colunas removidas, a duração das respostas).